# 🦀 Day 6: Error Handling in `main()`

---

In [ ]:
// main() can return Result!
// fn main() -> Result<(), Box<dyn std::error::Error>> {
//     let content = std::fs::read_to_string("config.txt")?;
//     let number: i32 = content.trim().parse()?;
//     println!("Config: {}", number);
//     Ok(())
// }
// If an error occurs, Rust prints it with Debug formatting

println!("main() → Result<(), Box<dyn Error>> enables ? throughout!");

In [ ]:
// Custom exit codes with std::process::exit
use std::process;

fn run_app() -> Result<(), Box<dyn std::error::Error>> {
    // Simulate app logic
    let config = "8080";
    let port: u16 = config.parse()?;
    println!("Server would start on port {}", port);
    Ok(())
}

// In real main():
// fn main() {
//     if let Err(e) = run_app() {
//         eprintln!("Error: {}", e);
//         process::exit(1);
//     }
// }

match run_app() {
    Ok(()) => println!("App ran successfully!"),
    Err(e) => println!("App error: {}", e),
}

In [ ]:
// Professional pattern: separate run() function

use std::fmt;

#[derive(Debug)]
enum CliError {
    MissingArg(String),
    InvalidArg { name: String, value: String },
    Io(std::io::Error),
}

impl fmt::Display for CliError {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        match self {
            CliError::MissingArg(a) => write!(f, "Missing argument: {}", a),
            CliError::InvalidArg { name, value } => 
                write!(f, "Invalid value '{}' for --{}", value, name),
            CliError::Io(e) => write!(f, "IO error: {}", e),
        }
    }
}

impl std::error::Error for CliError {}
impl From<std::io::Error> for CliError {
    fn from(e: std::io::Error) -> Self { CliError::Io(e) }
}

fn run(args: &[&str]) -> Result<(), CliError> {
    if args.is_empty() {
        return Err(CliError::MissingArg("filename".into()));
    }
    let filename = args[0];
    let content = std::fs::read_to_string(filename)?;  // io::Error → CliError
    println!("Read {} bytes", content.len());
    Ok(())
}

// Simulate main
let test_cases: Vec<Vec<&str>> = vec![
    vec![],                    // Missing arg
    vec!["nonexistent.txt"],   // IO error
];

for args in &test_cases {
    match run(args) {
        Ok(()) => println!("✅ Success"),
        Err(e) => {
            println!("❌ Error: {}", e);
            // In real app: process::exit(1);
        }
    }
}

In [ ]:
// Error reporting with chain
fn print_error_chain(mut err: &dyn std::error::Error) {
    println!("Error: {}", err);
    while let Some(source) = err.source() {
        println!("  Caused by: {}", source);
        err = source;
    }
}

// Demonstrating with a nested error
let io_err = std::io::Error::new(std::io::ErrorKind::NotFound, "file missing");
let cli_err = CliError::Io(io_err);
print_error_chain(&cli_err);

## 📋 Main Patterns Comparison

```rust
// Pattern 1: Simple — main returns Result
fn main() -> Result<(), Box<dyn Error>> {
    do_stuff()?;
    Ok(())
}

// Pattern 2: Custom error handling
fn main() {
    if let Err(e) = run() {
        eprintln!("Error: {}", e);
        std::process::exit(1);
    }
}

// Pattern 3: Full error chain reporting
fn main() {
    if let Err(e) = run() {
        eprintln!("Error: {}", e);
        let mut source = e.source();
        while let Some(cause) = source {
            eprintln!("  Caused by: {}", cause);
            source = cause.source();
        }
        std::process::exit(1);
    }
}
```

## 🏋️ Exercises

In [ ]:
// Exercise: Write a mini CLI app structure with:
// - Custom error type
// - run() function that parses args and processes a file
// - Proper error reporting with context

// YOUR CODE HERE


## 🎯 Key Takeaways

1. `fn main() -> Result<(), Box<dyn Error>>` enables `?` in main
2. Separate `run()` function for testability and clean error handling
3. Use `eprintln!()` for error output (goes to stderr)
4. `process::exit(1)` for non-zero exit codes
5. Print error chains with `.source()` for debugging

---
**Tomorrow:** Mini-project — CSV Parser! 📊